# 🎬 ليلة وناي وربابة — Stable Colab Montage

هذه النسخة مصممة للموبايل ولانقطاع جلسات Colab.

### ما المختلف؟
- تحفظ الخامات واللقطات داخل **Google Drive**.
- إذا فصل Colab، افتح الملف من جديد واضغط **Run all**؛ الموجود لن يُعاد تنزيله أو بناؤه.
- مرحلة الصوت تستخدم **WAV** لتفادي الخطأ السابق.
- لا يوجد تنزيل تلقائي للموبايل.
- في النهاية يظهر Preview داخل Colab فقط.


In [ ]:
# 1) ربط Google Drive وحفظ التقدم
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, shutil, os, time, requests
from IPython.display import Video, display

ROOT = Path('/content/drive/MyDrive/Leila_Nay_Rababa_Montage_Cache')
SRC = ROOT / 'sources'
SEG = ROOT / 'segments'
OUT = ROOT / 'output'
for p in (SRC, SEG, OUT):
    p.mkdir(parents=True, exist_ok=True)

print('✅ Cache:', ROOT)
print('✅ الملفات ستبقى محفوظة في Google Drive حتى لو فصلت الجلسة.')


In [ ]:
# 2) فحص FFmpeg
if shutil.which('ffmpeg') is None:
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq','ffmpeg'], check=True)

def run(cmd):
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0:
        print('\n--- FFmpeg error ---\n')
        print(p.stderr[-6000:])
        raise RuntimeError('FFmpeg failed')
    return p

print('✅ FFmpeg جاهز:', shutil.which('ffmpeg'))


In [ ]:
# 3) تنزيل الأغنية والخامات مرة واحدة فقط
SONG_URL = 'https://cdn.creativeclaw.co/u/2eb76212/audio/974df573-c163-4d17-81c1-94d92a4b4265.mp3'
SOURCE_URLS = {
  "rain_traffic": "https://cdn.creativeclaw.co/u/2eb76212/videos/fa51da72-d5cc-4978-be0f-a533733d6109.mp4",
  "rain_window": "https://cdn.creativeclaw.co/u/2eb76212/videos/ca11d0c5-8b6a-4563-8772-7dd8501c394e.mp4",
  "vertical_traffic": "https://cdn.creativeclaw.co/u/2eb76212/videos/24120068-d357-4d5c-90f2-2e6104fdc688.mp4",
  "friends_cafe": "https://cdn.creativeclaw.co/u/2eb76212/videos/53bd9e96-7b18-433b-b33a-a1c78f13e2af.mp4",
  "sunrise_city": "https://cdn.creativeclaw.co/u/2eb76212/videos/904668b0-3061-41e8-ba06-784fd7cdb0c0.mp4",
  "sunrise_road": "https://cdn.creativeclaw.co/u/2eb76212/videos/df967382-e534-4509-9772-3b27c25d81be.mp4"
}

def download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 100_000:
        print('⏭️ موجود مسبقًا:', dest.name, round(dest.stat().st_size/1024/1024, 1), 'MB')
        return
    print('⬇️ تنزيل:', dest.name)
    with requests.get(url, stream=True, timeout=180, headers={'User-Agent':'Mozilla/5.0'}) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + '.part')
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
        tmp.replace(dest)
    print('✅', dest.name, round(dest.stat().st_size/1024/1024, 1), 'MB')

download(SONG_URL, SRC/'song.mp3')
for name, url in SOURCE_URLS.items():
    download(url, SRC/f'{name}.mp4')

print('✅ كل المصادر متاحة.')


In [ ]:
# 4) خطة الـ50 ثانية
SHOTS = [('rain_window', 0.0, 1.05, 0.76, -0.045, 1.1, 3.90244), ('rain_traffic', 0.8, 0.92, 0.82, -0.04, 1.12, 3.90244), ('vertical_traffic', 0.0, 0.88, 0.86, -0.035, 1.1, 3.90244), ('rain_window', 2.0, 0.97, 0.8, -0.04, 1.11, 3.90244), ('rain_traffic', 3.0, 0.86, 0.88, -0.03, 1.1, 3.90244), ('friends_cafe', 0.0, 1.0, 0.96, -0.005, 1.05, 3.90244), ('friends_cafe', 2.0, 0.92, 1.0, 0.0, 1.05, 3.90244), ('vertical_traffic', 2.5, 0.88, 0.94, -0.02, 1.08, 3.90244), ('sunrise_city', 0.0, 1.03, 1.02, 0.0, 1.06, 3.90244), ('sunrise_road', 0.0, 0.98, 1.04, 0.005, 1.05, 3.90244), ('sunrise_city', 2.0, 0.94, 1.06, 0.008, 1.05, 3.90244), ('sunrise_road', 2.5, 0.9, 1.08, 0.01, 1.05, 3.90244), ('sunrise_road', 4.0, 1.03, 1.1, 0.012, 1.04, 3.17072)]
assert abs(sum(s[-1] for s in SHOTS) - 50.0) < 0.01
print('✅ عدد اللقطات:', len(SHOTS))


In [ ]:
# 5) بناء اللقطات — يتخطى أي لقطة تم إنشاؤها من قبل
for i, (src, start, speed, sat, bright, contrast, dur) in enumerate(SHOTS, start=1):
    out = SEG / f'{i:02d}.mp4'

    # اعتبر اللقطة مكتملة إذا الملف موجود وحجمه معقول
    if out.exists() and out.stat().st_size > 200_000:
        print(f'⏭️ Shot {i:02d} موجود')
        continue

    src_path = SRC / f'{src}.mp4'
    vf = (
        f"scale=720:1280:force_original_aspect_ratio=increase,"
        f"crop=720:1280,fps=30,setsar=1,"
        f"setpts={speed}*PTS,"
        f"eq=saturation={sat}:brightness={bright}:contrast={contrast},"
        f"format=yuv420p"
    )

    cmd = [
        'ffmpeg','-y','-hide_banner','-loglevel','error',
        '-stream_loop','-1','-ss',str(start),'-i',str(src_path),
        '-an','-vf',vf,'-t',f'{dur:.5f}',
        '-c:v','libx264','-preset','veryfast','-crf','21','-pix_fmt','yuv420p',
        str(out)
    ]
    run(cmd)
    print(f'✅ Shot {i:02d}/{len(SHOTS)}')

print('✅ كل اللقطات جاهزة ومخزنة في Drive.')


In [ ]:
# 6) تجميع الفيديو + قص الصوت بصيغة WAV (الإصلاح الأساسي)
concat_txt = ROOT / 'concat.txt'
with open(concat_txt, 'w') as f:
    for i in range(1, len(SHOTS)+1):
        f.write(f"file '{SEG / f'{i:02d}.mp4'}'\n")

video_concat = ROOT / 'video_concat.mp4'
if not video_concat.exists() or video_concat.stat().st_size < 500_000:
    run([
        'ffmpeg','-y','-hide_banner','-loglevel','error',
        '-f','concat','-safe','0','-i',str(concat_txt),
        '-an','-c','copy',str(video_concat)
    ])
    print('✅ تم تجميع الفيديو')
else:
    print('⏭️ الفيديو المجمّع موجود')

audio_wav = ROOT / 'audio_50.wav'
if not audio_wav.exists() or audio_wav.stat().st_size < 500_000:
    run([
        'ffmpeg','-y','-hide_banner','-loglevel','error',
        '-i',str(SRC/'song.mp3'),
        '-ss','252.8','-t','50',
        '-vn','-ac','2','-ar','44100','-c:a','pcm_s16le',
        str(audio_wav)
    ])
    print('✅ تم تجهيز الصوت WAV')
else:
    print('⏭️ الصوت المجهز موجود')


In [ ]:
# 7) إخراج Master و Preview — ويكمل من الموجود لو الجلسة فصلت
master = OUT / 'leila_nay_rababa_reel_50s_MASTER_720x1280.mp4'

if not master.exists() or master.stat().st_size < 1_000_000:
    run([
        'ffmpeg','-y','-hide_banner','-loglevel','error',
        '-i',str(video_concat),'-i',str(audio_wav),'-t','50',
        '-map','0:v:0','-map','1:a:0',
        '-c:v','libx264','-preset','veryfast','-crf','21',
        '-c:a','aac','-b:a','160k',
        '-pix_fmt','yuv420p','-movflags','+faststart',
        str(master)
    ])
    print('✅ MASTER تم')
else:
    print('⏭️ MASTER موجود')

preview = OUT / 'leila_nay_rababa_reel_50s_PREVIEW_540x960.mp4'

if not preview.exists() or preview.stat().st_size < 500_000:
    run([
        'ffmpeg','-y','-hide_banner','-loglevel','error',
        '-i',str(master),
        '-vf','scale=540:960',
        '-c:v','libx264','-preset','veryfast','-b:v','800k',
        '-c:a','aac','-b:a','96k',
        '-pix_fmt','yuv420p','-movflags','+faststart',
        str(preview)
    ])
    print('✅ PREVIEW تم')
else:
    print('⏭️ PREVIEW موجود')

print('\n🎬 انتهى المونتاج')
print('Preview:', round(preview.stat().st_size/1024/1024, 1), 'MB')
print('Master :', round(master.stat().st_size/1024/1024, 1), 'MB')


In [ ]:
# 8) معاينة داخل Colab فقط — لا تنزيل تلقائي
display(Video(str(preview), embed=True, width=320))


## التنزيل اختياري
بعد ما تشوف الـPreview لو عجبك، شغّل خلية واحدة فقط من الخليتين تحت.
**Run all لن ينزّل شيئًا للموبايل تلقائيًا** لأن الخلايا التالية تعرّف دوال فقط.


In [ ]:
# 9) جهّز زر تنزيل الـPreview — لا يبدأ التنزيل إلا إذا كتبت download_preview()
from google.colab import files

def download_preview():
    files.download(str(preview))

print('لو عايز Preview اكتب في خلية جديدة: download_preview()')


In [ ]:
# 10) جهّز زر تنزيل الـMaster — لا يبدأ التنزيل إلا إذا كتبت download_master()
def download_master():
    files.download(str(master))

print('لو عايز النسخة الكاملة اكتب في خلية جديدة: download_master()')
